In [ ]:
from src.generation import (
    vector_to_coordinate,
)
from src.spheroid import Spheroid
from src.projection import UVMap
from src.tectonic_sim import TectonicSimulation
import noise
from src.utils import coordinate_to_vector

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Line3DCollection, Poly3DCollection
import pyvista as pv
import panel as pn

pn.extension("vtk")
pv.set_jupyter_backend("trame")

In [ ]:
# grid_size = [7] * 3
# centering_translation = np.array([-size / 2 for size in grid_size], dtype=float)
# sampling_translation = np.array([4.0, -5.0, 0.0], dtype=float)
# arr = (
#     DataSeries.three_d_perlin(
#         *grid_size,
#         seed=42,
#         resolution=0.20,
#         scale=4.0,
#         translation=sampling_translation,
#         repeat=(255, 255, 255),
#     )
#     .translate_data(centering_translation - sampling_translation)
#     .cull_below_threshold(threshold=-0.25)
#     .cull_within_radius(radius=1, center_point=(0.0, 0.0, 0.0))
# )

# # only now split for plotting
# coords, values = split_points(arr.data)

# x, y, z = coords.T
# for rot_deg in range(0, 360, 45):
#     fig = plt.figure()
#     ax = fig.add_subplot(projection="3d")
#     sc = ax.scatter(x, y, z, c=values, cmap="plasma", s=20)
#     ax.view_init(elev=30, azim=rot_deg)
#     plt.colorbar(sc, ax=ax, label="Perlin value")
#     plt.show()

In [ ]:
# unit vectors from arr with coordinate_to_vector
# coordinate_to_vector only accepts one coordinate at a time,
# do no use raw python loops
# arr shape is [[z,y,z],value]
# from src.utils import coordinate_to_vector


# unit_vectors = np.array([coordinate_to_vector(coord) for coord in arr.data[:, 0]])
# coords, values = split_points(unit_vectors)
# x, y, z = coords.T
# for rot_deg in range(0, 360, 45):
#      fig = plt.figure()
#      ax = fig.add_subplot(projection="3d")
#      sc = ax.scatter(x, y, z, c=values, cmap="plasma", s=20)
#      ax.view_init(elev=30, azim=rot_deg)
#      plt.colorbar(sc, ax=ax, label="Perlin value")
#      plt.show()

In [ ]:
# ico = icosahedron(radius=1.0)
# vectors = np.stack(ico[0])

#

# vec_to_coord = np.vectorize(lambda d, m: vector_to_coordinate([d, m]), otypes=[np.ndarray])
# coords = np.vstack(vec_to_coord(vectors[:, 0], vectors[:, 1]))

# values = vectors[:, 1]
# print(coords.shape, values.shape)

# for rot_deg in range(0, 360, 45):
#      fig = plt.figure()
#      ax = fig.add_subplot(projection="3d")
#      sc = ax.scatter(coords[:, 0], coords[:, 1], coords[:, 2], c=values, cmap="plasma", s=20)
#      ax.view_init(elev=30, azim=rot_deg)
#      plt.colorbar(sc, ax=ax, label="Perlin value")
#      plt.show()

In [ ]:
radius = 10.0
subdivisions = 12
ico = Spheroid(radius=radius, subdivisions=subdivisions)
vectors, edges, faces = ico.vectors, ico.edges, ico.faces
# convert to ndarray form3)
vectors = np.stack(vectors)

# --- normalize all vectors in bulk ---
directions = np.stack(vectors[:, 0]).astype(float)
magnitudes = vectors[:, 1].astype(float)

norms = np.linalg.norm(directions, axis=1, keepdims=True)
safe_norms = np.where(norms == 0, 1.0, norms)
normalized_dirs = directions / safe_norms
normalized_mags = np.ones_like(magnitudes, dtype=float) * radius

vectors = np.empty((len(normalized_dirs), 2), dtype=object)
vectors[:, 0] = [d.astype(np.float32) for d in normalized_dirs]
vectors[:, 1] = normalized_mags

# --- convert to coordinates in bulk ---
coords = normalized_dirs
values = normalized_mags

ico.vectors = vectors

# --- plotting ---
fig = plt.figure()
ax = fig.add_subplot(projection="3d")

# scatter vertices
sc = ax.scatter(
    coords[:, 0],
    coords[:, 1],
    coords[:, 2],
    c=values,
    cmap="plasma",
    s=50,
    depthshade=True,
)

# edges
edge_lines = np.stack([coords[edges[:, 0]], coords[edges[:, 1]]], axis=1)
ax.add_collection3d(Line3DCollection(edge_lines, colors="black", linewidths=1))

# faces
face_polys = [coords[face] for face in faces]
ax.add_collection3d(
    Poly3DCollection(
        face_polys, facecolors="cyan", edgecolors="k", linewidths=0.5, alpha=0.2
    )
)

ax.view_init(elev=30, azim=90)
ax.set_box_aspect([1, 1, 1])

plt.colorbar(sc, ax=ax, label="Magnitude")
plt.show()

In [ ]:
resolution = (512, 512)
file_path = f"uv_{subdivisions}_{resolution[0]}x{resolution[1]}.json"
if Path(file_path).exists():
    uv_map = UVMap.load(file_path)
else:
    uv_map = UVMap.from_spheroid(resolution, ico, uv_padding=0.125)
    uv_map.save(file_path)

In [ ]:
# plot the unwrapped faces and their connections
projection = uv_map.projection
faces_2d, vertices_2d, vertex_map = (
    projection.faces,
    projection.vertices,
    projection.vertex_map,
)
plt.figure(figsize=(4, 4))
for face in faces_2d:
    poly = vertices_2d[np.array(face, dtype=int)]
    plt.fill(poly[:, 0], poly[:, 1], edgecolor="black", fill=False, linewidth=0.5)
plt.gca().set_aspect("equal", adjustable="box")
plt.title("Unwrapped faces of spheroid")
plt.xlabel("X")
plt.ylabel("Y")
plt.show()

In [ ]:
creation_configs = {
    "density_range": (0.35, 1.15),
    "velocity_range": (-3.0, 3.0),
    "height_range": (-0.75, 0.75),
    "velocity_decay": 0.07,
    "density_decay": 0.03,
}

setup_configs = {
    "velocity_amplification": 1.0,
    "friction_dampening": 1.2,
    "contributing_neighbor_radius": 3,
    "base_friction": 1.0,
    "position_weight": 0.8,
    "velocity_weight": 0.75,
    "density_weight": 0.5,
}

run_configs = {
    "stress_thresholds": np.arange(0.3, 1.2, 0.3),
    "stress_propagation_radius_per_threshold": 3,
    "stress_distribution_factor": 0.24,
}

region_count = 8

# --- simulation ---
techtonic_sim = TectonicSimulation.from_spheroid(
    ico, region_count, seed=257, **creation_configs
)
techtonic_sim.setup_simulation(**setup_configs)
techtonic_sim.run(cycles=10, **run_configs)
planetoid = techtonic_sim.apply_to_spheroid(ico.clone(), stress_scalar=0.75)

coords = np.asarray(planetoid.vertices, dtype=float)
edges_array = np.asarray(
    planetoid.edges, dtype=int
)  # kept if you want edge overlays later
faces_index_arrays = [
    np.asarray(face_indices, dtype=int) for face_indices in planetoid.faces
]

In [ ]:
# --- per-face scalars (nodes correspond 1:1 to faces) ---
face_stress_values = np.array(
    [node.stress for node in techtonic_sim.nodes], dtype=float
)
face_region_values = np.array([node.region for node in techtonic_sim.nodes], dtype=int)
min_stress_value = float(face_stress_values.min())
max_stress_value = float(face_stress_values.max())

# --- pack polygon faces for PyVista (no triangulation) ---
faces_packed_parts = []
for face_indices in faces_index_arrays:
    vertex_count_in_face = np.array([len(face_indices)], dtype=np.int64)
    faces_packed_parts.append(vertex_count_in_face)
    faces_packed_parts.append(face_indices.astype(np.int64))
faces_packed = np.concatenate(faces_packed_parts)

# --- build polydata ---
poly_mesh = pv.PolyData(coords, faces=faces_packed)
poly_mesh.cell_data.clear()
poly_mesh.cell_data["stress"] = face_stress_values
poly_mesh.cell_data["region"] = face_region_values

In [ ]:
# --- interactive backend (PyVista 0.43+ valid backends) ---
pv.set_jupyter_backend(
    "trame"
)  # options: "static", "client", "server", "trame", "html", "none"

# --- plotter and initial actor ---
plotter = pv.Plotter()
actor = plotter.add_mesh(
    poly_mesh,
    scalars="stress",
    cmap="viridis",
    clim=(min_stress_value, max_stress_value),
    show_edges=True,
    edge_color="black",
    smooth_shading=False,
)
plotter.add_axes()

# --- UI callbacks (update actor in place) ---


def on_scalar_change(data_set: bool):
    if data_set:
        selected_scalar = "region"
        color_map = "plasma"
        scalar_range = (0.0, float(region_count - 1))
    else:
        selected_scalar = "stress"
        color_map = "viridis"
        scalar_range = (min_stress_value, max_stress_value)
    poly_mesh.set_active_scalars(selected_scalar)
    actor.mapper.scalar_range = scalar_range
    actor.mapper.lookup_table = pv.LookupTable(color_map)
    actor.mapper.SetScalarModeToUseCellData()
    plotter.update_scalar_bar_range(scalar_range)
    plotter.render()


# plot the widget near the top-middle
plotter.add_checkbox_button_widget(
    on_scalar_change, value=False, position=(0, plotter.window_size[1] / 2 - 40)
)

plotter.show()

In [ ]:
projection_configs = {
    # "weights_exponent": 1.0,
    "plane_relaxation": 0.5,
    "scaler": 1.0,
}


def noise_wrapper(x, y, z):
    return noise.pnoise3(
        x,
        y,
        z,
        repeatx=1024,
        repeaty=1024,
        repeatz=1024,
        base=42,
        octaves=5,
        persistence=0.9,
        lacunarity=0.4,
    )


uv_map.generate_noise(noise_function=noise_wrapper, spheroid=ico, **projection_configs)
uv_map_2 = UVMap.load(f"uv_{subdivisions}_{resolution[0]}x{resolution[1]}.json")


def noise_wrapper(x, y, z):
    return noise.pnoise3(
        x,
        y,
        z,
        repeatx=1024,
        repeaty=1024,
        repeatz=1024,
        base=42,
        octaves=12,
        persistence=1.1,
        lacunarity=1.1,
    )


uv_map_2.generate_noise(
    noise_function=noise_wrapper, spheroid=ico, **projection_configs
)


# --- Gradient-based scaling of noise values ---
# v = 1 / (1 + k * m), where m = max slope to first valid neighbor in each direction
# If a neighbor pixel along a direction is missing, step outward (with wrap) until a valid one is found.

# --- Gradient-based scaling on uv_map_2 ---
steepness_k = 1.0  # controls attenuation strength

width, height = uv_map_2.resolution
coords_scaled = np.asarray(uv_map_2.coords, dtype=int)  # (N2, 2)
values_scaled = np.asarray(uv_map_2.values, dtype=float)  # (N2,)

coord_to_value_scaled: dict[tuple[int, int], float] = {
    (int(u_coord), int(v_coord)): float(val)
    for (u_coord, v_coord), val in zip(coords_scaled, values_scaled)
}

direction_steps = [
    (-1, -1),
    (-1, 0),
    (-1, 1),
    (0, -1),
    (0, 1),
    (1, -1),
    (1, 0),
    (1, 1),
]
max_search_steps = max(width, height)

scaled_values_with_gradient = np.empty_like(values_scaled)

for point_index, (u_coord, v_coord) in enumerate(coords_scaled):
    center_value = values_scaled[point_index]
    max_slope = 0.0
    for delta_u, delta_v in direction_steps:
        for step_distance in range(1, max_search_steps + 1):
            neighbor_u = (u_coord + delta_u * step_distance) % width
            neighbor_v = (v_coord + delta_v * step_distance) % height
            neighbor_key = (int(neighbor_u), int(neighbor_v))
            if neighbor_key in coord_to_value_scaled:
                neighbor_value = coord_to_value_scaled[neighbor_key]
                slope = abs(neighbor_value - center_value) / float(step_distance)
                if slope > max_slope:
                    max_slope = slope
                break
    scale_factor = 1.0 / (1.0 + steepness_k * max_slope)
    scaled_values_with_gradient[point_index] = center_value * scale_factor

# write back to uv_map_2
uv_map_2.values = scaled_values_with_gradient + uv_map.values / 2

In [ ]:
# rasterize: this gives pixel coordinates + face indices
pixel_coords, face_indices = uv_map.coords, uv_map.face_indices
pixel_values = uv_map.values


# Unpack for plotting
us = pixel_coords[:, 0]
vs = pixel_coords[:, 1]
vals = pixel_values


plt.figure(figsize=(6, 6))
plt.scatter(us, vs, c=vals, s=1, vmin=-1, vmax=1, cmap="viridis")
plt.gca().set_aspect("equal", adjustable="box")
plt.colorbar(label="Value")
plt.xlabel("U")
plt.ylabel("V")
plt.title("UV raster of spheroid")
plt.show()

In [ ]:
projection = uv_map.projection
ocean = ico.clone()

# --- project UV pixels to 3D on the deformed planetoid ---
coords_3d = np.asarray(
    [
        projection.project_to_spheroid(
            int(u), int(v), int(face_index), planetoid, **projection_configs
        )
        for (u, v), face_index in zip(uv_map.coords, uv_map.face_indices)
    ],
    dtype=float,
)

# per-pixel values (terrain)
values = uv_map.values
noise_scalar = 1.0
scatter_coords = []
scatter_magnitudes = []
for pixel_index, point_xyz in enumerate(coords_3d):
    vector = coordinate_to_vector(point_xyz)
    vector[1] += values[pixel_index] * noise_scalar
    scatter_magnitudes.append(vector[1])
    scatter_coords.append(vector_to_coordinate(vector))
scatter_coords = np.asarray(scatter_coords, dtype=float)
scatter_magnitudes = np.asarray(scatter_magnitudes, dtype=float)

# --- planetoid mesh geometry (for overlay) ---
planetoid_vectors = planetoid.vectors
planetoid_edges = np.asarray(planetoid.edges, dtype=np.int64)
planetoid_faces = [np.asarray(face, dtype=np.int64) for face in planetoid.faces]
planetoid_mesh_coords = np.asarray(
    [vector_to_coordinate(vec) for vec in planetoid_vectors], dtype=float
)

# --- ocean surface (undeformed spheroid) ---
ocean_vectors = np.stack(ocean.vectors)
ocean_edges = np.asarray(ocean.edges, dtype=np.int64)
ocean_directions = np.stack(ocean_vectors[:, 0]).astype(float)
ocean_magnitudes = ocean_vectors[:, 1].astype(float)
ocean_coords = ocean_directions * ocean_magnitudes[:, None]
ocean_faces = [np.asarray(face, dtype=np.int64) for face in ocean.faces]

# ===================== PACKING (kept separate) =====================

# planetoid faces (polygon connectivity)
planetoid_faces_packed = np.hstack(
    [np.concatenate(([len(idx)], idx)) for idx in planetoid_faces]
).astype(np.int64)

# planetoid edges as VTK lines
planetoid_line_cells = np.c_[
    np.full(len(planetoid_edges), 2, dtype=np.int64), planetoid_edges
].ravel()

# ocean faces (polygon connectivity)
ocean_faces_packed = np.hstack(
    [np.concatenate(([len(idx)], idx)) for idx in ocean_faces]
).astype(np.int64)

ocean_line_cells = np.c_[
    np.full(len(ocean_edges), 2, dtype=np.int64), ocean_edges
].ravel()

# ===================== RANGES / CAMERA =====================

magnitude_range = (float(scatter_magnitudes.min()), float(scatter_magnitudes.max()))
ocean_mean_radius = float(ocean_magnitudes.mean())
ocean_clim = (0.98 * ocean_mean_radius, 1.02 * ocean_mean_radius)

scene_extent_points = np.vstack((planetoid_mesh_coords, scatter_coords, ocean_coords))
scene_radius = 3.0 * float(np.linalg.norm(scene_extent_points, axis=1).max())

In [ ]:
pv.set_jupyter_backend("trame")

# --- build polydata objects ---
ocean_mesh = pv.PolyData(ocean_coords, faces=ocean_faces_packed)
ocean_mesh.point_data["Elevation"] = ocean_magnitudes.astype(float)

planetoid_edges_mesh = pv.PolyData(planetoid_mesh_coords, lines=planetoid_line_cells)
ocean_edge_mesh = pv.PolyData(ocean_coords, lines=ocean_line_cells)

terrain_points = pv.PolyData(scatter_coords)
terrain_points["magnitude"] = scatter_magnitudes.astype(float)

# --- single interactive plot ---
plotter = pv.Plotter()

# ocean: dark blue, semi-transparent
# plotter.add_mesh(
#     ocean_mesh,
#     # scalars="Elevation",
#     cmap="Blues",
#     # clim=ocean_clim,
#     opacity=0.50,
#     show_edges=False,
#     name="ocean",
#     # scalar_bar_args={"title": "Ocean Elevation"},
# )

# planetoid wireframe
# plotter.add_mesh(
#     planetoid_edges_mesh,
#     color="black",
#     line_width=1.5,
#     name="planetoid_edges",
# )

# ocean wireframe
plotter.add_mesh(
    ocean_edge_mesh,
    color="darkblue",
    line_width=0.5,
    name="ocean_edges",
)

# deformed terrain points
plotter.add_mesh(
    terrain_points,
    scalars="magnitude",
    cmap="viridis",
    clim=magnitude_range,
    point_size=6,
    render_points_as_spheres=True,
    name="terrain_points",
    scalar_bar_args={"title": "Elevation"},
)

# camera similar to elev=30°, azim=45°
elev_rad = np.deg2rad(30.0)
azim_rad = np.deg2rad(45.0)
plotter.camera_position = [
    (
        scene_radius * np.cos(elev_rad) * np.cos(azim_rad),
        scene_radius * np.cos(elev_rad) * np.sin(azim_rad),
        scene_radius * np.sin(elev_rad),
    ),
    (0.0, 0.0, 0.0),
    (0.0, 0.0, 1.0),
]

plotter.show()

In [ ]:
# default_value = -10.0
# vectors, edges, faces = apply_dataseries_to_polygon(
#     dataseries=arr.data,
#     polygon=(ico.vectors, ico.edges, ico.faces),
#     default=default_value,
#     # ray_radius=0.5,
#     # ray_step=0.25,
#     # max_march=2.5
# )
# print("completed applying data series on polygon...")
# r = 1.0
# scaled_vectors = np.array(
#     [np.array([v[0], r, v[2]], dtype=object) for v in vectors],
#     dtype=object,
# )
# print("completed scaling vectors...")

# # convert to coordinates using helper
# coords = np.stack(
#     [vector_to_coordinate(v[:2], origin_point=(0, 0, 0)) for v in scaled_vectors]
# )
# values = np.array([v[2] for v in scaled_vectors], dtype=float)
# print("completed converting to coordinates...")

# # build face geometry and average colors
# face_coords = [coords[f] for f in faces]
# face_colors = np.array([values[f].mean() for f in faces])
# min_legend_value = min(-0.5, face_colors.min())
# print("completed building face geometry and colors...")

# print("Start plotting...")
# for rot_deg in range(0, 360, 90):
#     fig = plt.figure()
#     ax = fig.add_subplot(projection="3d")

#     # edges
#     edge_lines = np.stack([coords[edges[:, 0]], coords[edges[:, 1]]], axis=1)
#     ax.add_collection3d(Line3DCollection(edge_lines, colors="black", linewidths=1))
#     norm = mpl.colors.Normalize(vmin=min_legend_value, vmax=face_colors.max())
#     facecolors = plt.cm.plasma(norm(face_colors))
#     ax.add_collection3d(
#         Poly3DCollection(
#             face_coords,
#             facecolors=facecolors,
#             edgecolors="k",
#             linewidths=0.5,
#             alpha=1.0,
#         )
#     )

#     ax.view_init(elev=30, azim=rot_deg)
#     ax.set_box_aspect([1, 1, 1])

#     mappable = mpl.cm.ScalarMappable(cmap="plasma")
#     mappable.set_array(face_colors)
#     # set the min value on the legend to -5 or face_colors whichever is lower
#     # use the min_legend_value as the minimum for the colorbar
#     plt.colorbar(mappable, ax=ax, label="Face mean value")

#     plt.show()